# Group 16 | V2 Acceptance on Tesla T4

Nghiệm thu CLI V2 và CUDA timing của **P7-T02**, đồng thời thu bằng chứng cho **P7-T04**. Notebook build source tích hợp, đối chiếu V0/V1/V2 trên dataset cuối kỳ, chạy bộ hồi quy và Compute Sanitizer.

## Quick Start and Required Files

1. Chọn **Runtime → Change runtime type → T4 GPU**, sau đó **Run all**.
2. Cell cấu hình mặc định thử GitHub public ở ref `refs/heads/chuong/p7`. Nếu ref chưa có đủ source, upload `build/Group16_P7_Source.zip` khi notebook hỏi. Để chỉ dùng GitHub, chọn `github` và điền SHA đã publish. Để dùng ngay gói local, chọn `zip`.
3. Notebook ưu tiên gói dataset cuối kỳ trên Drive, rồi thử tải ảnh gốc để chuẩn bị lại nếu cần. Khi Drive giới hạn tải, cung cấp `dataset.zip` có `manifest.csv`. Nếu thiếu ảnh nguồn synthetic, cung cấp ảnh nguồn của nhóm hoặc `synthetic.zip` có đủ sáu cặp.
4. Đọc bảng, xem gallery challenge và tải `Group16_P7_T04_Evidence_*.zip`. P7-T04 chuyển **Chờ review** khi các cổng tự động đạt và đã đối chiếu ảnh challenge. Trạng thái **Xong** được ghi sau khi PR nghiệm thu merge.

**Nội dung:** [Runtime](#runtime-verification) · [Source](#source-snapshot) · [Dataset](#dataset-and-fixtures) · [Execution](#acceptance-execution) · [Results](#result-matrix) · [Gallery](#panorama-gallery) · [Memcheck](#memcheck-evidence) · [Export](#evidence-export)

<a id="runtime-verification"></a>

## 1. Runtime and Settings

Chọn nguồn theo bản muốn nghiệm thu. Mỗi lần chạy khởi tạo workspace mới và kiểm tra GPU, driver, nvcc, Compute Sanitizer trước khi xử lý dataset.

In [ ]:
# @title Verification Settings { display-mode: "form" }
SOURCE_MODE = "auto"  # @param ["auto", "github", "zip"]
GIT_URL = "https://github.com/nh-atuan/panorama-stitching.git"
GIT_REF = "refs/heads/chuong/p7"  # @param {type:"string"}
SOURCE_ZIP = "/content/Group16_P7_Source.zip"  # @param {type:"string"}
DATASET_DRIVE_URL = "https://drive.google.com/drive/folders/1_J_q5q-K1g7ZNnczTWvXBzwpOsof9ghL?usp=sharing"  # @param {type:"string"}
DATASET_ZIP = "/content/dataset.zip"  # @param {type:"string"}
SYNTHETIC_ZIP = "/content/synthetic.zip"  # @param {type:"string"}
SYNTHETIC_SOURCE = ""  # @param {type:"string"}

In [ ]:
# @title Verification Helpers { display-mode: "form" }
from pathlib import Path, PurePosixPath
from datetime import datetime, timezone
from google.colab import files
import json
import stat
import subprocess
import sys
import tempfile
import zipfile

WORK = Path(tempfile.mkdtemp(prefix="group16_p7_", dir="/content"))
EVIDENCE = WORK / "evidence"

def call(args, cwd=None):
    subprocess.run([str(arg) for arg in args], cwd=cwd, check=True)

def upload_one(prompt):
    print(prompt)
    uploaded = files.upload()
    if len(uploaded) != 1:
        raise ValueError("Upload exactly one file.")
    return Path(next(iter(uploaded))).resolve()

def unpack(archive, destination):
    destination.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(archive) as source:
        for member in source.infolist():
            name = PurePosixPath(member.filename)
            if (name.is_absolute() or ".." in name.parts or chr(92) in member.filename
                    or stat.S_ISLNK(member.external_attr >> 16)):
                raise ValueError(f"Unsafe archive entry: {member.filename}")
        source.extractall(destination)

In [ ]:
# @title Check Tesla T4 and CUDA Toolkit
gpu_output = subprocess.check_output(
    ["nvidia-smi", "--query-gpu=name,driver_version", "--format=csv,noheader"], text=True
).strip()
print(gpu_output)
if not any(line.split(",")[0].strip() in {"Tesla T4", "NVIDIA T4", "T4"}
           for line in gpu_output.splitlines()):
    raise RuntimeError("Select a Tesla T4 runtime before continuing.")
call(["nvcc", "--version"])
call(["compute-sanitizer", "--version"])

<a id="source-snapshot"></a>

## 2. Source Snapshot and Integration Preflight

GitHub được tải vào workspace riêng và chốt ở commit thực tế. Chế độ `auto` dùng ZIP nếu ref GitHub tải lỗi hoặc thiếu các file tích hợp. Chế độ `github` dừng với lỗi để giữ đúng ref đã chọn. Gói ZIP cung cấp Git base và hash snapshot, còn harness sẽ kiểm tra lại hash từng file trước khi build.

In [ ]:
# @title Resolve and Verify Source
REQUIRED_FILES = (
    "src/main.cu",
    "src/v2/v2_pipeline.h",
    "src/v2/v2_pipeline.cu",
    "scripts/verify_v2_dataset.py",
    "tests/gpu/test_gpu_v2_pipeline.cu",
)

def require_integration(project):
    missing = [name for name in REQUIRED_FILES if not (project / name).is_file()]
    if missing:
        raise ValueError("Missing integrated files: " + ", ".join(missing))

def git_checkout(project, url, ref):
    if not ref.strip():
        raise ValueError("GIT_REF is required for GitHub.")
    project.mkdir()
    call(["git", "init", "--quiet"], cwd=project)
    result = subprocess.run(
        ["git", "fetch", "--depth", "1", "--", url, ref],
        cwd=project, capture_output=True, text=True, timeout=120,
    )
    if result.returncode:
        raise ValueError(result.stderr.strip() or "Git fetch failed.")
    call(["git", "checkout", "--detach", "--quiet", "FETCH_HEAD"], cwd=project)
    require_integration(project)
    return subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=project, text=True).strip()

if SOURCE_MODE not in {"auto", "github", "zip"}:
    raise ValueError("SOURCE_MODE must be auto, github, or zip.")
PROJECT = None
if SOURCE_MODE != "zip":
    git_project = WORK / "source_git"
    try:
        git_commit = git_checkout(git_project, GIT_URL, GIT_REF)
    except (ValueError, subprocess.SubprocessError, OSError) as error:
        if SOURCE_MODE == "github":
            raise
        print(f"GitHub source unavailable for {GIT_REF}: {error}")
        print("Using the local source archive.")
    else:
        PROJECT = git_project
        print(f"Source: GitHub {GIT_URL}")
        print(f"Requested ref: {GIT_REF}")
        print(f"Resolved commit: {git_commit}")

if PROJECT is None:
    source_zip = Path(SOURCE_ZIP)
    if not source_zip.is_file():
        source_zip = upload_one("Upload Group16_P7_Source.zip from the integrated workspace.")
    PROJECT = WORK / "source_zip"
    unpack(source_zip, PROJECT)
    require_integration(PROJECT)
    manifest = json.loads((PROJECT / "source_manifest.json").read_text(encoding="utf-8"))
    print(f"Source: {source_zip.name}")
    print(f"Git base: {manifest['git_commit']}")
    print(f"Source snapshot SHA-256: {manifest['snapshot_sha256']}")
print("Production source preflight passed.")

<a id="dataset-and-fixtures"></a>

## 3. Dataset and Fixture Metadata

Bộ ảnh chính thức gồm `thanh-standard1`, `thanh-standard2`, `de-standard2` và `de-challenge1`. Harness đối chiếu 24 ảnh theo tên cặp, kích thước và SHA-256 trong manifest. Sáu fixture synthetic gồm tịnh tiến và phối cảnh ở ba độ phân giải. Cell dưới hiển thị metadata gốc của từng fixture trước khi chạy.

In [ ]:
# @title Prepare Official Data and Synthetic Fixtures
def find_one(directory, filename):
    matches = list(directory.rglob(filename))
    if len(matches) != 1:
        raise ValueError(f"Expected one {filename} in {directory}, found {len(matches)}")
    return matches[0]

dataset_zip = Path(DATASET_ZIP)
if not dataset_zip.is_file():
    try:
        from importlib.util import find_spec
        if find_spec("gdown") is None:
            call([sys.executable, "-m", "pip", "install", "-q", "gdown"])
        import gdown
        downloaded = WORK / "dataset_drive"
        downloaded.mkdir()
        gdown.download_folder(url=DATASET_DRIVE_URL, output=str(downloaded) + "/", use_cookies=False)
        dataset_zip = find_one(downloaded, "dataset.zip")
    except Exception as error:
        print(f"Official Drive ZIP unavailable: {error}")
if not dataset_zip.is_file():
    try:
        call([sys.executable, "scripts/fetch_and_prepare_dataset.py", "--out", PROJECT / "dataset"], cwd=PROJECT)
    except subprocess.CalledProcessError:
        dataset_zip = upload_one("Drive download failed. Upload the official dataset ZIP.")
if dataset_zip.is_file():
    extracted = WORK / "dataset_upload"
    unpack(dataset_zip, extracted)
    DATASET = find_one(extracted, "manifest.csv").parent
else:
    DATASET = PROJECT / "dataset"

synthetic_zip = Path(SYNTHETIC_ZIP)
source = Path(SYNTHETIC_SOURCE) if SYNTHETIC_SOURCE else PROJECT / "raw/tuan/standard1_left.jpg"
if not synthetic_zip.is_file() and not source.is_file() and not SYNTHETIC_SOURCE:
    try:
        from importlib.util import find_spec
        if find_spec("gdown") is None:
            call([sys.executable, "-m", "pip", "install", "-q", "gdown"])
        import gdown
        raw = PROJECT / "raw"
        raw.mkdir(exist_ok=True)
        gdown.download_folder(
            url="https://drive.google.com/drive/folders/127SOMhoY2EwBbkCxGWGPju-EbmG0QiOq?usp=sharing",
            output=str(raw) + "/", use_cookies=False,
        )
    except Exception as error:
        print(f"P7 synthetic source unavailable on Drive: {error}")
if not synthetic_zip.is_file() and not source.is_file():
    uploaded = upload_one("Upload the original synthetic source image or a synthetic fixtures ZIP.")
    if zipfile.is_zipfile(uploaded):
        synthetic_zip = uploaded
    else:
        source = uploaded
if synthetic_zip.is_file():
    extracted = WORK / "synthetic_upload"
    unpack(synthetic_zip, extracted)
    fixtures = [path for path in extracted.rglob("small_truth.json") if path.parent.name == "translation"]
    if len(fixtures) != 1:
        raise ValueError("The archive must contain one translation/small_truth.json.")
    SYNTHETIC = fixtures[0].parent.parent
else:
    SYNTHETIC = WORK / "synthetic"
    call([sys.executable, "scripts/make_synthetic_pair.py", "--source", source, "--out", SYNTHETIC], cwd=PROJECT)
print(f"Dataset: {DATASET}")
print(f"Synthetic fixtures: {SYNTHETIC}")

In [ ]:
# @title Inspect Dataset Manifest and Synthetic Ground Truth
import csv
import hashlib
from IPython.display import Markdown, display

manifest_path = DATASET / "manifest.csv"
with manifest_path.open(encoding="utf-8", newline="") as stream:
    image_manifest = list(csv.DictReader(stream))
print(f"Dataset manifest: {len(image_manifest)} image records")
print(f"Dataset manifest SHA-256: {hashlib.sha256(manifest_path.read_bytes()).hexdigest()}")
lines = ["| Fixture | Dimensions | Direction | Source | Source SHA-256 |",
         "|---|---:|---|---|---|"]
for kind in ("translation", "projective"):
    for size in ("small", "medium", "large"):
        truth_path = SYNTHETIC / kind / f"{size}_truth.json"
        truth = json.loads(truth_path.read_text(encoding="utf-8"))
        name = f"{kind}/{size}"
        lines.append(f"| {name} | {truth['width']}×{truth['height']} | `{truth['direction']}` | "
                     f"`{truth.get('source', 'unrecorded')}` | `{truth.get('source_sha256', 'unrecorded')}` |")
display(Markdown("\n".join(lines)))

<a id="acceptance-execution"></a>

## 4. Build and Acceptance Execution

Harness build `nvcc -arch=sm_75 -O2 --fmad=false` và chạy pipeline production. Mỗi CLI thành công có một warm-up và năm lần đo. V2 được kiểm tra schema 11 cột, hai dòng tổng hợp, mã lỗi và D2H khi canvas lớn hơn ảnh A. Phép đo GPU dùng CUDA events cho từng stage và kernel, cùng clock đơn điệu cho end-to-end đã đồng bộ. Hai lượt sanitizer chạy riêng khỏi các lượt timing.

In [ ]:
# @title Build and Run the Acceptance Harness
command = [sys.executable, str(PROJECT / "scripts/verify_v2_dataset.py"),
           "--dataset", str(DATASET), "--synthetic-dir", str(SYNTHETIC),
           "--out", str(EVIDENCE)]
ACCEPTANCE = subprocess.run(command, cwd=PROJECT)
print(f"Acceptance exit code: {ACCEPTANCE.returncode}. The next cells retain evidence on failure.")

<a id="result-matrix"></a>

## 5. Result Matrix and CLI Gates

Bảng và các cổng lấy trực tiếp từ report do harness sinh. Với ca thành công, đọc theo thứ tự trạng thái, độ lệch chiếu, canvas và RGB. Với challenge, so sánh trạng thái V0/V1/V2 và kiểm tra ảnh ở phần tiếp theo.

In [ ]:
# @title Display the Generated Verification Report
report_path = EVIDENCE / "p7-v2-verification.md"
summary_path = EVIDENCE / "verification_summary.json"
if not report_path.is_file() or not summary_path.is_file():
    raise RuntimeError("The harness did not write a report and summary. Inspect the execution cell.")
summary = json.loads(summary_path.read_text(encoding="utf-8"))
display(Markdown(report_path.read_text(encoding="utf-8")))

<a id="panorama-gallery"></a>

## 6. Input Pairs and Panorama Gallery

Ba site chuẩn hiển thị A/B và panorama V2 ở cỡ medium để đối chiếu đầu vào với kết quả. Với `de-challenge1`, hiển thị cả ba cỡ, các panorama V0/V1/V2 có sẵn và trạng thái từ log. Trạng thái thành công chỉ cho biết chương trình xuất được ảnh. Phần review cần xem đường nối và sai lệch hình học, nhất là challenge/small.

In [ ]:
# @title Review Standard Panoramas and Challenge Outputs
from PIL import Image
import matplotlib.pyplot as plt

cases = {(case["site"], case["size"]): case for case in summary.get("cases", [])}

def show_pair(site, size, versions):
    case = cases.get((site, size))
    if case is None:
        return
    items = [
        (f"Input {label}", DATASET / site / f"{size}_{side}.jpg", "Input unavailable")
        for label, side in (("A", "left"), ("B", "right"))
    ]
    items += [
        (f"V{version}", EVIDENCE / "panoramas" / site / size / f"v{version}.png",
         case["versions"][version]["status"])
        for version in versions
    ]
    figure, axes = plt.subplots(1, len(items), figsize=(3.6 * len(items), 4.1))
    figure.suptitle(f"{case['pair']} / {size}")
    for axis, (title, image_path, status) in zip(axes, items):
        axis.set_title(title)
        axis.axis("off")
        if image_path.is_file():
            with Image.open(image_path) as image:
                image.thumbnail((960, 540))
                axis.imshow(image)
        else:
            axis.text(0.5, 0.5, status, ha="center", va="center", wrap=True)
    plt.show()
    plt.close(figure)

for site in ("site1", "site2", "site3"):
    show_pair(site, "medium", (2,))
for size in ("small", "medium", "large"):
    show_pair("challenge", size, (0, 1, 2))

<a id="memcheck-evidence"></a>

## 7. Memcheck and Reproducibility Summary

Kiểm tra riêng suite production và CLI V2 với `site1/small`. Mỗi target cần mã thoát 0, 0 lỗi và 0 byte rò rỉ. Ghi lại GPU, Git base, hash source và dataset trước khi đối chiếu hoặc chia sẻ kết quả.

In [ ]:
# @title Inspect Memcheck and Provenance
environment = summary.get("environment", {})
source = summary.get("source", {})
print(f"GPU and driver: {environment.get('gpu', 'unrecorded')}")
print(f"Git base: {source.get('git_commit', 'unrecorded')}")
print(f"Source SHA-256: {source.get('snapshot_sha256', 'unrecorded')}")
print(f"Dataset manifest SHA-256: {summary.get('manifest_sha256', 'unrecorded')}")
print(f"Real configurations recorded: {len(cases)}/12")
for label, result in summary.get("memcheck", {}).items():
    print(f"{label}: {'PASS' if result['passed'] else 'FAIL'} (exit {result['returncode']})")
    for line in result["lines"]:
        print(line)
if not summary.get("memcheck"):
    print("No memcheck results recorded.")

<a id="evidence-export"></a>

## 8. Evidence Export and Review

ZIP lưu log gốc, báo cáo, JSON từng cấu hình và panorama challenge. Panorama chuẩn còn ở runtime, hash nằm trong JSON từng ca. Sau khi các cổng đạt, xem lại gallery, đính kèm ZIP vào PR và cập nhật báo cáo cùng task board.

In [ ]:
# @title Download the Evidence Archive
stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
archive = WORK / f"Group16_P7_T04_Evidence_{stamp}.zip"
with zipfile.ZipFile(archive, "w", compression=zipfile.ZIP_DEFLATED) as package:
    for path in sorted(EVIDENCE.rglob("*")):
        rel = path.relative_to(EVIDENCE)
        if not path.is_file() or (rel.parts[0] == "panoramas" and rel.parts[1] != "challenge"):
            continue
        package.write(path, rel.as_posix())
print(f"Evidence ZIP: {archive}")
files.download(str(archive))
if ACCEPTANCE.returncode != 0 or not summary["passed"]:
    raise RuntimeError("Acceptance failed. The downloaded archive contains the evidence.")
print("Automated gates passed. Review the challenge gallery before submitting.")

## References and Scope

Notebook dùng harness của P7-T04 và source tích hợp P7-T02. Benchmark/profiling tối ưu và APOD của P7-T03 có bộ kết quả riêng. [Public GitHub repository](https://github.com/nh-atuan/panorama-stitching) · [CUDA Event Management](https://docs.nvidia.com/cuda/cuda-runtime-api/cuda_runtime_api/group__CUDART__EVENT.html)